In [1]:
# Cell 1 — Install dependencies
%pip install psycopg2-binary sqlalchemy pandas

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 36.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
import psycopg2

conn = psycopg2.connect(host="bench-cockroach-25009.j77.aws-ap-south-1.cockroachlabs.cloud", port=26257, dbname="defaultdb", user="avyaansh", password="KUM4UGIHawG0UD5MbNXg3Q", sslmode="verify-full", sslrootcert="system")

conn.autocommit = True
cursor = conn.cursor()
print("✅ Connected!")

✅ Connected!


In [9]:
# CockroachDB Notes:
# - STRING is native and preferred over VARCHAR
# - DECIMAL works as-is
# - TIMESTAMP works as-is
# - Self-referencing FK on categories is supported

schema_sql = """
-- Drop tables in reverse dependency order if re-running
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS cart_items;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS users;
DROP TABLE IF EXISTS categories;

-- 1. categories (self-referencing FK)
CREATE TABLE categories (
    category_id INT         PRIMARY KEY,
    name        STRING      NOT NULL,
    parent_id   INT         NULL,
    CONSTRAINT fk_category_parent FOREIGN KEY (parent_id)
        REFERENCES categories (category_id)
);

-- 2. products
CREATE TABLE products (
    product_id  INT         PRIMARY KEY,
    name        STRING      NOT NULL,
    category_id INT         NOT NULL,
    price       DECIMAL     NOT NULL,
    stock_qty   INT         NOT NULL,
    description STRING,
    created_at  TIMESTAMP   NOT NULL,
    CONSTRAINT fk_product_category FOREIGN KEY (category_id)
        REFERENCES categories (category_id)
);

-- 3. users
CREATE TABLE users (
    user_id     INT         PRIMARY KEY,
    email       STRING      UNIQUE NOT NULL,
    name        STRING      NOT NULL,
    created_at  TIMESTAMP   NOT NULL
);

-- 4. cart_items
CREATE TABLE cart_items (
    cart_item_id INT        PRIMARY KEY,
    user_id      INT        NOT NULL,
    product_id   INT        NOT NULL,
    quantity     INT        NOT NULL,
    added_at     TIMESTAMP  NOT NULL,
    UNIQUE (user_id, product_id),
    CONSTRAINT fk_cart_user    FOREIGN KEY (user_id)    REFERENCES users    (user_id),
    CONSTRAINT fk_cart_product FOREIGN KEY (product_id) REFERENCES products (product_id)
);

-- 5. orders
CREATE TABLE orders (
    order_id     INT        PRIMARY KEY,
    user_id      INT        NOT NULL,
    status       STRING     NOT NULL,
    total_amount DECIMAL    NOT NULL,
    created_at   TIMESTAMP  NOT NULL,
    CONSTRAINT fk_order_user FOREIGN KEY (user_id) REFERENCES users (user_id)
);

-- 6. order_items
CREATE TABLE order_items (
    order_item_id INT       PRIMARY KEY,
    order_id      INT       NOT NULL,
    product_id    INT       NOT NULL,
    quantity      INT       NOT NULL,
    unit_price    DECIMAL   NOT NULL,
    CONSTRAINT fk_oi_order   FOREIGN KEY (order_id)   REFERENCES orders   (order_id),
    CONSTRAINT fk_oi_product FOREIGN KEY (product_id) REFERENCES products (product_id)
);
"""

# Execute all statements
for statement in schema_sql.strip().split(';'):
    statement = statement.strip()
    if statement:
        cursor.execute(statement)

print("✅ Schema created successfully!")

✅ Schema created successfully!


In [10]:
index_sql = """
CREATE INDEX IF NOT EXISTS idx_products_category_id  ON products    (category_id);
CREATE INDEX IF NOT EXISTS idx_products_name         ON products    (name);
CREATE INDEX IF NOT EXISTS idx_products_price        ON products    (price);
CREATE INDEX IF NOT EXISTS idx_cart_items_user_id    ON cart_items  (user_id);
CREATE INDEX IF NOT EXISTS idx_orders_user_id        ON orders      (user_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON order_items (product_id);
"""

for statement in index_sql.strip().split(';'):
    statement = statement.strip()
    if statement:
        cursor.execute(statement)

print("✅ All indexes created successfully!")

✅ All indexes created successfully!


In [12]:
import os
import csv
import time
from psycopg2.extras import execute_values

DATA_DIR = r"C:\Users\avyaa\ecommerce_data"

def load_csv(table_name, filename, columns, batch_size=500):
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = [tuple(row[col] if row[col] != '' else None for col in columns) for row in reader]

    col_str      = ", ".join(columns)
    insert_sql   = f"INSERT INTO {table_name} ({col_str}) VALUES %s"
    total        = len(rows)
    loaded       = 0
    start        = time.time()

    for i in range(0, total, batch_size):
        batch = rows[i:i + batch_size]
        execute_values(cursor, insert_sql, batch)
        loaded += len(batch)
        print(f"  {table_name}: {loaded}/{total} rows loaded...", end='\r')

    elapsed = round(time.time() - start, 2)
    print(f"  ✅ {table_name}: {total} rows loaded in {elapsed}s          ")

print("🚀 Loading data...\n")

load_csv("categories",  "categories.csv",  ["category_id", "name", "parent_id"])
load_csv("products",    "products.csv",    ["product_id", "name", "category_id", "price", "stock_qty", "description", "created_at"])
load_csv("users",       "users.csv",       ["user_id", "email", "name", "created_at"])
load_csv("cart_items",  "cart_items.csv",  ["cart_item_id", "user_id", "product_id", "quantity", "added_at"])
load_csv("orders",      "orders.csv",      ["order_id", "user_id", "status", "total_amount", "created_at"])
load_csv("order_items", "order_items.csv", ["order_item_id", "order_id", "product_id", "quantity", "unit_price"])

print("\n✅ All tables loaded!")

🚀 Loading data...

  ✅ categories: 50 rows loaded in 0.04s          
  ✅ products: 10000 rows loaded in 7.07s          
  ✅ users: 5000 rows loaded in 2.23s          
  ✅ cart_items: 8000 rows loaded in 4.36s          
  ✅ orders: 3000 rows loaded in 2.01s          
  ✅ order_items: 9000 rows loaded in 4.79s          

✅ All tables loaded!


In [13]:
import time
import random
import statistics
from collections import defaultdict
from tqdm.notebook import tqdm

# ============================================
# QUERY FUNCTIONS
# ============================================

def search_products(keyword):
    """Search products by keyword with pagination"""
    offset = random.randint(0, 500)
    cursor.execute("""
        SELECT product_id, name, price
        FROM products
        WHERE name ILIKE %s
        LIMIT 20 OFFSET %s
    """, (f'%{keyword}%', offset))
    return cursor.fetchall()

def browse_category(category_id, min_price, max_price, in_stock_only):
    """Browse products in a category with filters"""
    offset = random.randint(0, 200)
    stock_filter = "AND stock_qty > 0" if in_stock_only else ""
    cursor.execute(f"""
        SELECT p.product_id, p.name, p.price, p.stock_qty
        FROM products p
        WHERE p.category_id = %s
          AND p.price BETWEEN %s AND %s
          {stock_filter}
        ORDER BY p.created_at DESC
        LIMIT 20 OFFSET %s
    """, (category_id, min_price, max_price, offset))
    return cursor.fetchall()

def product_detail(product_id):
    """Get product details (2 queries: product info + sold count)"""
    # Query 1: Product with category
    cursor.execute("""
        SELECT p.product_id, p.name, p.price, p.stock_qty, p.description, c.name AS category_name
        FROM products p
        JOIN categories c ON c.category_id = p.category_id
        WHERE p.product_id = %s
    """, (product_id,))
    product = cursor.fetchone()
    
    # Query 2: Count sold
    cursor.execute("""
        SELECT COALESCE(SUM(quantity), 0) AS total_sold
        FROM order_items
        WHERE product_id = %s
    """, (product_id,))
    sold = cursor.fetchone()
    
    return product, sold

def add_to_cart(user_id, product_id, quantity):
    """Add product to cart (read stock + upsert)"""
    # Check stock
    cursor.execute("SELECT stock_qty FROM products WHERE product_id = %s", (product_id,))
    stock = cursor.fetchone()
    if not stock or stock[0] < quantity:
        raise Exception("Insufficient stock")
    
    # Upsert cart item
    cursor.execute("""
        INSERT INTO cart_items (cart_item_id, user_id, product_id, quantity, added_at)
        VALUES ((SELECT COALESCE(MAX(cart_item_id), 0) + 1 FROM cart_items), %s, %s, %s, NOW())
        ON CONFLICT (user_id, product_id)
        DO UPDATE SET quantity = cart_items.quantity + EXCLUDED.quantity
    """, (user_id, product_id, quantity))

def update_or_remove_cart(user_id, product_id):
    """Update cart quantity or remove item"""
    if random.random() < 0.5:
        # Update quantity
        new_qty = random.randint(1, 3)
        cursor.execute("""
            UPDATE cart_items
            SET quantity = %s
            WHERE user_id = %s AND product_id = %s
        """, (new_qty, user_id, product_id))
    else:
        # Remove item
        cursor.execute("""
            DELETE FROM cart_items
            WHERE user_id = %s AND product_id = %s
        """, (user_id, product_id))

def checkout(user_id):
    """Complete checkout transaction (ACID)"""
    try:
        conn.autocommit = False  # Start transaction
        
        # Get cart items
        cursor.execute("""
            SELECT ci.product_id, ci.quantity, p.price
            FROM cart_items ci
            JOIN products p ON p.product_id = ci.product_id
            WHERE ci.user_id = %s
        """, (user_id,))
        cart = cursor.fetchall()
        
        if not cart:
            conn.rollback()
            conn.autocommit = True
            return
        
        # Calculate total
        total_amount = sum(qty * price for _, qty, price in cart)
        
        # Create order
        cursor.execute("""
            INSERT INTO orders (order_id, user_id, status, total_amount, created_at)
            VALUES ((SELECT COALESCE(MAX(order_id), 0) + 1 FROM orders), %s, 'pending', %s, NOW())
            RETURNING order_id
        """, (user_id, total_amount))
        order_id = cursor.fetchone()[0]
        
        # Insert order items & update stock
        for product_id, quantity, price in cart:
            cursor.execute("""
                INSERT INTO order_items (order_item_id, order_id, product_id, quantity, unit_price)
                VALUES ((SELECT COALESCE(MAX(order_item_id), 0) + 1 FROM order_items), %s, %s, %s, %s)
            """, (order_id, product_id, quantity, price))
            
            cursor.execute("""
                UPDATE products
                SET stock_qty = stock_qty - %s
                WHERE product_id = %s AND stock_qty >= %s
            """, (quantity, product_id, quantity))
            
            if cursor.rowcount == 0:
                raise Exception("Stock check failed during checkout")
        
        # Clear cart
        cursor.execute("DELETE FROM cart_items WHERE user_id = %s", (user_id,))
        
        conn.commit()
        conn.autocommit = True
        
    except Exception as e:
        conn.rollback()
        conn.autocommit = True
        raise e

def payment_completion(order_id):
    """Mark order as completed (idempotent)"""
    cursor.execute("""
        UPDATE orders
        SET status = 'completed'
        WHERE order_id = %s AND status = 'pending'
    """, (order_id,))

# ============================================
# WORKLOAD GENERATOR
# ============================================

def generate_workload():
    """Generate randomized workload based on distribution"""
    operations = []
    
    # Search (25% = 2,500)
    keywords = ['Pro', 'Ultra', 'Premium', 'Smart', 'Classic', 'Modern', 'Advanced', 'Essential']
    operations.extend([('search', random.choice(keywords)) for _ in range(2500)])
    
    # Browse Category (30% = 3,000)
    operations.extend([
        ('browse_category', random.randint(1, 50), random.uniform(10, 100), random.uniform(100, 500), random.choice([True, False]))
        for _ in range(3000)
    ])
    
    # Product Detail (38% = 3,800)
    operations.extend([('product_detail', random.randint(1, 10000)) for _ in range(3800)])
    
    # Add to Cart (3.5% = 350)
    operations.extend([
        ('add_to_cart', random.randint(1, 5000), random.randint(1, 10000), random.randint(1, 3))
        for _ in range(350)
    ])
    
    # Update/Remove Cart (1.5% = 150)
    operations.extend([
        ('update_cart', random.randint(1, 5000), random.randint(1, 10000))
        for _ in range(150)
    ])
    
    # Checkout (1.2% = 120)
    operations.extend([('checkout', random.randint(1, 5000)) for _ in range(120)])
    
    # Payment Completion (0.8% = 80)
    operations.extend([('payment', random.randint(1, 3000)) for _ in range(80)])
    
    # CRITICAL: Randomize execution order
    random.shuffle(operations)
    
    return operations

# ============================================
# EXECUTION ENGINE
# ============================================

def execute_operation(op):
    """Execute a single operation and return latency (ms) or error"""
    op_type = op[0]
    args = op[1:]
    
    start = time.perf_counter()
    try:
        if op_type == 'search':
            search_products(*args)
        elif op_type == 'browse_category':
            browse_category(*args)
        elif op_type == 'product_detail':
            product_detail(*args)
        elif op_type == 'add_to_cart':
            add_to_cart(*args)
        elif op_type == 'update_cart':
            update_or_remove_cart(*args)
        elif op_type == 'checkout':
            checkout(*args)
        elif op_type == 'payment':
            payment_completion(*args)
        
        latency = (time.perf_counter() - start) * 1000  # Convert to ms
        return op_type, latency, None
    
    except Exception as e:
        latency = (time.perf_counter() - start) * 1000
        return op_type, latency, str(e)

def run_workload(workload, desc="Running"):
    """Execute workload and collect metrics"""
    results = defaultdict(lambda: {'latencies': [], 'errors': 0})
    
    for op in tqdm(workload, desc=desc, ncols=100):
        op_type, latency, error = execute_operation(op)
        results[op_type]['latencies'].append(latency)
        if error:
            results[op_type]['errors'] += 1
    
    return results

# ============================================
# METRIC CALCULATION
# ============================================

def calculate_metrics(results, total_time_sec):
    """Calculate performance metrics"""
    metrics = {}
    
    for op_type, data in results.items():
        latencies = data['latencies']
        error_count = data['errors']
        total_ops = len(latencies)
        
        if not latencies:
            continue
        
        avg_latency = statistics.mean(latencies)
        p95_latency = statistics.quantiles(latencies, n=20)[18] if len(latencies) > 1 else latencies[0]
        p99_latency = statistics.quantiles(latencies, n=100)[98] if len(latencies) > 1 else latencies[0]
        throughput = total_ops / total_time_sec
        error_rate = (error_count / total_ops) * 100
        
        metrics[op_type] = {
            'ops': total_ops,
            'avg_ms': round(avg_latency, 2),
            'p95_ms': round(p95_latency, 2),
            'p99_ms': round(p99_latency, 2),
            'qps': round(throughput, 2),
            'error_rate': round(error_rate, 2)
        }
    
    # Overall metrics
    all_latencies = [lat for data in results.values() for lat in data['latencies']]
    all_errors = sum(data['errors'] for data in results.values())
    
    metrics['OVERALL'] = {
        'ops': len(all_latencies),
        'avg_ms': round(statistics.mean(all_latencies), 2),
        'p95_ms': round(statistics.quantiles(all_latencies, n=20)[18] if len(all_latencies) > 1 else all_latencies[0], 2),
        'p99_ms': round(statistics.quantiles(all_latencies, n=100)[98] if len(all_latencies) > 1 else all_latencies[0], 2),
        'qps': round(len(all_latencies) / total_time_sec, 2),
        'error_rate': round((all_errors / len(all_latencies)) * 100, 2)
    }
    
    return metrics

def print_metrics(metrics, iteration=None):
    """Print metrics in table format"""
    header = f"\n{'='*95}\n"
    if iteration:
        header += f"  ITERATION {iteration} RESULTS\n{'='*95}\n"
    else:
        header += f"  BENCHMARK RESULTS\n{'='*95}\n"
    
    print(header)
    print(f"{'Operation':<20} {'Ops':<8} {'Avg(ms)':<10} {'P95(ms)':<10} {'P99(ms)':<10} {'QPS':<10} {'Err%':<8}")
    print("="*95)
    
    # Sort by operation type, OVERALL last
    sorted_ops = sorted([k for k in metrics.keys() if k != 'OVERALL']) + ['OVERALL']
    
    for op_type in sorted_ops:
        m = metrics[op_type]
        print(f"{op_type:<20} {m['ops']:<8} {m['avg_ms']:<10} {m['p95_ms']:<10} {m['p99_ms']:<10} {m['qps']:<10} {m['error_rate']:<8}")
    
    print("="*95)

# ============================================
# WARMUP & BENCHMARK RUNNERS
# ============================================

def run_warmup():
    """Execute warmup phase (500 read-only operations)"""
    print("\n🔥 WARMUP PHASE\n")
    warmup_ops = []
    
    # 200 searches
    keywords = ['Pro', 'Ultra', 'Premium', 'Smart']
    warmup_ops.extend([('search', random.choice(keywords)) for _ in range(200)])
    
    # 200 browse
    warmup_ops.extend([
        ('browse_category', random.randint(1, 50), 10.0, 500.0, True)
        for _ in range(200)
    ])
    
    # 100 product details
    warmup_ops.extend([('product_detail', random.randint(1, 10000)) for _ in range(100)])
    
    random.shuffle(warmup_ops)
    
    run_workload(warmup_ops, desc="Warming up")
    print("\n✅ Warmup complete\n")

def run_benchmark(iterations=3):
    """Execute main benchmark with multiple iterations"""
    print(f"\n🚀 MAIN BENCHMARK ({iterations} iterations, 10,000 ops each)\n")
    
    all_iteration_metrics = []
    
    for i in range(1, iterations + 1):
        print(f"\n--- Iteration {i}/{iterations} ---")
        
        # Generate fresh randomized workload
        workload = generate_workload()
        
        # Execute
        start_time = time.perf_counter()
        results = run_workload(workload, desc=f"Iteration {i}")
        total_time = time.perf_counter() - start_time
        
        # Calculate metrics
        metrics = calculate_metrics(results, total_time)
        all_iteration_metrics.append(metrics)
        
        # Print iteration results
        print_metrics(metrics, iteration=i)
    
    # Aggregate summary
    print(f"\n{'='*95}")
    print(f"  SUMMARY ACROSS {iterations} ITERATIONS")
    print(f"{'='*95}\n")
    
    print(f"Total operations executed: {iterations * 10000}")
    
    # Average OVERALL metrics across iterations
    overall_avg_ms = statistics.mean([m['OVERALL']['avg_ms'] for m in all_iteration_metrics])
    overall_p95_ms = statistics.mean([m['OVERALL']['p95_ms'] for m in all_iteration_metrics])
    overall_p99_ms = statistics.mean([m['OVERALL']['p99_ms'] for m in all_iteration_metrics])
    overall_qps = statistics.mean([m['OVERALL']['qps'] for m in all_iteration_metrics])
    overall_err = statistics.mean([m['OVERALL']['error_rate'] for m in all_iteration_metrics])
    
    print(f"Average latency:       {overall_avg_ms:.2f} ms")
    print(f"Average P95 latency:   {overall_p95_ms:.2f} ms")
    print(f"Average P99 latency:   {overall_p99_ms:.2f} ms")
    print(f"Average throughput:    {overall_qps:.2f} QPS")
    print(f"Average error rate:    {overall_err:.2f} %")
    print(f"\n{'='*95}\n")

print("✅ Benchmark engine loaded successfully!")
print("\nNext steps:")
print("  1. Run: run_warmup()")
print("  2. Run: run_benchmark(iterations=3)")

✅ Benchmark engine loaded successfully!

Next steps:
  1. Run: run_warmup()
  2. Run: run_benchmark(iterations=3)


In [14]:
run_warmup()


🔥 WARMUP PHASE



Warming up:   0%|                                                           | 0/500 [00:00<?, ?it/s]


✅ Warmup complete



In [15]:
run_benchmark(iterations=3)


🚀 MAIN BENCHMARK (3 iterations, 10,000 ops each)


--- Iteration 1/3 ---


Iteration 1:   0%|                                                        | 0/10000 [00:00<?, ?it/s]


  ITERATION 1 RESULTS

Operation            Ops      Avg(ms)    P95(ms)    P99(ms)    QPS        Err%    
add_to_cart          350      88.08      95.33      145.63     0.43       6.29    
browse_category      3000     53.01      82.67      106.81     3.65       0.0     
checkout             120      347.48     557.74     782.33     0.15       11.67   
payment              80       41.6       44.07      58.72      0.1        0.0     
product_detail       3800     87.66      95.74      148.27     4.62       0.0     
search               2500     97.18      115.01     345.91     3.04       0.0     
update_cart          150      41.33      42.63      60.98      0.18       0.0     
OVERALL              10000    81.71      107.83     350.16     12.15      0.36    

--- Iteration 2/3 ---


Iteration 2:   0%|                                                        | 0/10000 [00:00<?, ?it/s]


  ITERATION 2 RESULTS

Operation            Ops      Avg(ms)    P95(ms)    P99(ms)    QPS        Err%    
add_to_cart          350      86.93      93.29      139.67     0.42       7.14    
browse_category      3000     53.11      83.13      107.07     3.64       0.0     
checkout             120      338.69     578.58     901.26     0.15       16.67   
payment              80       43.1       44.13      186.14     0.1        0.0     
product_detail       3800     87.93      91.86      173.19     4.61       0.0     
search               2500     97.83      116.94     345.39     3.03       0.0     
update_cart          150      41.32      43.62      53.3       0.18       0.0     
OVERALL              10000    81.87      106.13     349.18     12.14      0.45    

--- Iteration 3/3 ---


Iteration 3:   0%|                                                        | 0/10000 [00:00<?, ?it/s]


  ITERATION 3 RESULTS

Operation            Ops      Avg(ms)    P95(ms)    P99(ms)    QPS        Err%    
add_to_cart          350      88.18      93.54      266.44     0.43       8.57    
browse_category      3000     53.33      84.45      110.83     3.68       0.0     
checkout             120      312.84     605.65     778.32     0.15       12.5    
payment              80       41.45      44.52      106.73     0.1        0.0     
product_detail       3800     86.05      87.99      145.47     4.66       0.0     
search               2500     98.55      115.07     343.84     3.06       0.0     
update_cart          150      41.24      42.41      72.53      0.18       0.0     
OVERALL              10000    81.13      105.29     342.96     12.25      0.45    

  SUMMARY ACROSS 3 ITERATIONS

Total operations executed: 30000
Average latency:       81.57 ms
Average P95 latency:   106.42 ms
Average P99 latency:   347.43 ms
Average throughput:    12.18 QPS
Average error rate:    0.42 %




In [16]:
cursor.close()
conn.close()
print("🔒 Connection closed.")

🔒 Connection closed.
